In [ ]:
pip install yfinance

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Define el ticker de la acción y el índice del mercado
ticker = 'WMT'
market_index = '^GSPC'  # S&P 500 como proxy del mercado

# Descargar datos históricos
data = yf.download([ticker, market_index], start="2020-01-01", end="2021-01-01")

# Calcular los retornos logarítmicos
log_returns = np.log(data['Adj Close'] / data['Adj Close'].shift(1))
log_returns = log_returns.dropna()

# Preparar la variable independiente y dependiente
X = log_returns[market_index]  # Retornos del mercado
y = log_returns[ticker]  # Retornos de la acción
X = sm.add_constant(X)  # añadir una constante al modelo

# Ajustar el modelo de regresión
model = sm.OLS(y, X)
results = model.fit()

# Imprimir el resumen del modelo
print(results.summary())

[*********************100%%**********************]  2 of 2 completed

                            OLS Regression Results                            
Dep. Variable:                    WMT   R-squared:                       0.317
Model:                            OLS   Adj. R-squared:                  0.314
Method:                 Least Squares   F-statistic:                     116.0
Date:                Wed, 15 May 2024   Prob (F-statistic):           1.78e-22
Time:                        16:48:30   Log-Likelihood:                 681.04
No. Observations:                 252   AIC:                            -1358.
Df Residuals:                     250   BIC:                            -1351.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0005      0.001      0.529      0.5

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm




def descargar_datos(tickers, start_date, end_date):
    """Descarga los datos ajustados al cierre y maneja errores individuales de ticker."""
    datos = pd.DataFrame()
    for ticker in tickers:
        try:
            ticker_data = yf.download(ticker, start=start_date, end=end_date)['Adj Close']
            datos[ticker] = ticker_data
        except Exception as e:
            print(f"Error descargando datos para {ticker}: {e}")
    return datos

def calcular_modelo(datos, market_index):
    """Calcula el modelo de regresión para cada ticker y devuelve un DataFrame con los resultados."""
    resultados = pd.DataFrame()
    log_returns = np.log(datos / datos.shift(1))
    log_returns.dropna(inplace=True)

    for ticker in datos.columns:
        if ticker != market_index:
            try:
                X = sm.add_constant(log_returns[market_index])  # Mercado como variable independiente
                y = log_returns[ticker]  # Retornos del ticker como variable dependiente
                modelo = sm.OLS(y, X).fit()
                resultados.loc[ticker, 'Const'] = modelo.params['const']
                resultados.loc[ticker, 'P-valor Const'] = modelo.pvalues['const']
                resultados.loc[ticker, 'Beta'] = modelo.params[market_index]
                resultados.loc[ticker, 'P-valor Beta'] = modelo.pvalues[market_index]
                resultados.loc[ticker, 'R^2'] = modelo.rsquared
            except Exception as e:
                print(f"Error al ajustar el modelo para {ticker}: {e}")
    return resultados

# Define la lista de tickers y el índice del mercado
tickers = ["MSFT", "AAPL", "NVDA", "AMZN", "META", "GOOGL", "GOOG", "BRK.B", "LLY", "AVGO", "JPM", "TSLA", "XOM", "V", "UNH", "MA", "PG", "JNJ", "HD", "MRK", "COST", "ABBV", "CRM", "CVX", "AMD", "NFLX", "BAC", "WMT", "PEP", "KO", "LIN", "TMO", "ADBE", "DIS", "ACN", "WFC", "ORCL", "CSCO", "MCD", "QCOM", "ABT", "CAT", "INTU", "AMAT", "IBM", "VZ", "GE", "CMCSA", "NOW", "INTC", "DHR", "COP", "UBER", "TXN", "PFE", "UNP", "AMGN", "PM", "LOW", "SPGI", "ISRG", "MU", "RTX", "GS", "NEE", "HON", "ETN", "AXP", "LRCX", "BKNG", "PGR", "T", "ELV", "SYK", "C", "MS", "PLD", "BLK", "MDT", "TJX", "NKE", "UPS", "SCHW", "DE", "CI", "BA", "VRTX", "BMY", "CB", "ADP", "MMC", "BSX", "REGN", "SBUX", "ADI", "LMT", "FI", "KLAC", "CVS", "BX", "MDLZ", "AMT", "SNPS", "GILD", "PANW", "CDNS", "TMUS", "CMG", "MPC", "EOG", "ICE", "TGT", "SHW", "SLB", "CME", "SO", "ZTS", "WM", "ANET", "DUK", "MO", "EQIX", "PH", "PSX", "CL", "ITW", "FCX", "PYPL", "CSX", "BDX", "MCK", "ABNB", "APH", "TT", "TDG", "USB", "GD", "ORLY", "EMR", "HCA", "NOC", "PNC", "PCAR", "AON", "FDX", "PXD", "NXPI", "MAR", "MCO", "VLO", "CEG", "CTAS", "MSI", "ROP", "ECL", "NSC", "EW", "COF", "AIG", "DXCM", "HLT", "AZO", "APD", "F", "TRV", "AJG", "ADSK", "TFC", "GM", "WELL", "MMM", "NUE", "SPG", "CPRT", "CARR", "MCHP", "URI", "ROST", "WMB", "DHI", "SMCI", "OKE", "PSA", "NEM", "OXY", "MET", "AFL", "ALL", "TEL", "GWW", "SRE", "O", "AEP", "IQV", "JCI", "AMP", "FTNT", "CCI", "MSCI", "DLR", "FAST", "FIS", "BK", "HES", "STZ", "IDXX", "KMB", "A", "DOW", "AME", "PRU", "LULU", "LEN", "MNST", "CMI", "D", "CTVA", "ODFL", "OTIS", "COR", "PAYX", "LHX", "GIS", "HUM", "CNC", "SYY", "RSG", "MLM", "CSGP", "PWR", "IR", "YUM", "EXC", "GEHC", "FANG", "IT", "HAL", "KR", "PCG", "VMC", "CTSH", "KMI", "GEV", "ACGL", "MRNA", "KVUE", "DG", "BKR", "DVN", "CDW", "EL", "ADM", "GPN", "PEG", "PPG", "VRSK", "DD", "RCL", "MPWR", "ROK", "KDP", "EA", "EFX", "EXR", "DFS", "ED", "HIG", "VICI", "FICO", "XYL", "DAL", "ANSS", "XEL", "BIIB", "FTV", "ON", "KHC", "HSY", "WST", "CBRE", "MTD", "KEYS", "WTW", "RMD", "EIX", "CHTR", "TSCO", "CAH", "WAB", "EBAY", "DLTR", "ZBH", "LYB", "TROW", "AVB", "HWM", "TRGP", "WEC", "HPQ", "WY", "NVR", "CHD", "PHM", "BLDR", "FITB", "DOV", "GLW", "RJF", "TTWO", "BR", "NDAQ", "STT", "WDC", "MTB", "HPE", "AWK", "IRM", "SBAC", "GRMN", "ALGN", "DECK", "DTE", "STLD", "ETR", "HUBB", "ULTA", "PTC", "MOH", "CPAY", "NTAP", "AXON", "EQR", "IFF", "APTV", "BAX", "GPC", "CTRA", "STE", "BALL", "ES", "ILMN", "INVH", "BRO", "PPL", "HBAN", "WAT", "FE", "ARE", "COO", "TDY", "LVS", "CBOE", "VLTO", "FSLR", "CINF", "AEE", "TXT", "MKC", "RF", "WBD", "DRI", "PFG", "J", "OMC", "NTRS", "HOLX", "IEX", "CLX", "CNP", "LH", "JBL", "WRB", "LDOS", "AVY", "EXPE", "SYF", "DPZ", "TYL", "VTR", "MAS", "ATO", "CMS", "MRO", "STX", "EXPD", "PKG", "LUV", "TSN", "FDS", "NRG", "SWKS", "VRSN", "TER", "EG", "CE", "CFG", "AKAM", "JBHT", "CCL", "ENPH", "ESS", "BBY", "SNA", "TRMB", "ALB", "BG", "EPAM", "MAA", "POOL", "CF", "ZBRA", "K", "EQT", "CAG", "SWK", "NDSN", "LYV", "DGX", "HST", "KEY", "UAL", "VTRS", "L", "LKQ", "WBA", "PNR", "DOC", "IP", "AMCR", "KMX", "RVTY", "CRL", "MGM", "ROL", "GEN", "JKHY", "WRK", "LNT", "KIM", "TAP", "AES", "EVRG", "IPG", "EMN", "SJM", "PODD", "JNPR", "ALLE", "FFIV", "HII", "UDR", "LW", "QRVO", "NI", "CPT", "TECH", "APA", "AOS", "BBWI", "MOS", "UHS", "CTLT", "INCY", "TFX", "WYNN", "HRL", "TPR", "PAYC", "NWSA", "REG", "DAY", "AIZ", "HSIC", "SOLV", "MTCH", "GL", "BF.B", "CZR", "AAL", "BXP", "CPB", "MKTX", "CHRW", "PNW", "GNRC", "BWA", "NCLH", "RHI", "ETSY", "FOXA", "BEN", "IVZ", "FMC", "FRT", "HAS", "DVA", "CMA", "BIO", "RL", "MHK",]

market_index = "^GSPC"

# Descarga de datos con manejo de errores
data = descargar_datos(tickers + [market_index], "2023-01-01", "2024-01-01")

# Asegurarse de que los datos se hayan descargado correctamente antes de continuar
if not data.empty:
    resultados = calcular_modelo(data, market_index)
    print(resultados)
else:
    print("No se obtuvieron datos válidos, revisa la lista de tickers y la conexión a internet.")

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['BRK.B']: Exception('%ticker%: No timezone found, symbol may be delisted')
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%*******************

Error al ajustar el modelo para MSFT: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para AAPL: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para NVDA: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para AMZN: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para META: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para GOOGL: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para GOOG: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para BRK.B: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para LLY: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para AVGO

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os

def descargar_datos(ticker, start_date, end_date):
    """Descarga los datos ajustados al cierre para un ticker específico y maneja errores."""
    try:
        ticker_data = yf.download(ticker, start=start_date, end=end_date)['Adj Close']
        return ticker_data
    except Exception as e:
        print(f"Error descargando datos para {ticker}: {e}")
        return None

def calcular_modelo(ticker_data, market_data):
    """Calcula el modelo de regresión para un ticker y devuelve los resultados."""
    log_returns_ticker = np.log(ticker_data / ticker_data.shift(1)).dropna()
    log_returns_market = np.log(market_data / market_data.shift(1)).dropna()
    merged_data = pd.concat([log_returns_ticker, log_returns_market], axis=1).dropna()

    if merged_data.empty:
        return None

    X = sm.add_constant(merged_data.iloc[:, 1])  # Mercado como variable independiente
    y = merged_data.iloc[:, 0]  # Retornos del ticker como variable dependiente

    modelo = sm.OLS(y, X).fit()
    resultados = {
        'Const': modelo.params['const'],
        'P-valor Const': modelo.pvalues['const'],
        'Beta': modelo.params[market_data.name],
        'P-valor Beta': modelo.pvalues[market_data.name],
        'R^2': modelo.rsquared
    }
    return resultados

# Define la lista de tickers y el índice del mercado
tickers = ["MSFT", "AAPL", "NVDA", "AMZN", "META", "GOOGL", "GOOG", "BRK.B", "LLY", "AVGO", "JPM", "TSLA", "XOM", "V", "UNH", "MA", "PG", "JNJ", "HD", "MRK", "COST", "ABBV", "CRM", "CVX", "AMD", "NFLX", "BAC", "WMT", "PEP", "KO", "LIN", "TMO", "ADBE", "DIS", "ACN", "WFC", "ORCL", "CSCO", "MCD", "QCOM", "ABT", "CAT", "INTU", "AMAT", "IBM", "VZ", "GE", "CMCSA", "NOW", "INTC", "DHR", "COP", "UBER", "TXN", "PFE", "UNP", "AMGN", "PM", "LOW", "SPGI", "ISRG", "MU", "RTX", "GS", "NEE", "HON", "ETN", "AXP", "LRCX", "BKNG", "PGR", "T", "ELV", "SYK", "C", "MS", "PLD", "BLK", "MDT", "TJX", "NKE", "UPS", "SCHW", "DE", "CI", "BA", "VRTX", "BMY", "CB", "ADP", "MMC", "BSX", "REGN", "SBUX", "ADI", "LMT", "FI", "KLAC", "CVS", "BX", "MDLZ", "AMT", "SNPS", "GILD", "PANW", "CDNS", "TMUS", "CMG", "MPC", "EOG", "ICE", "TGT", "SHW", "SLB", "CME", "SO", "ZTS", "WM", "ANET", "DUK", "MO", "EQIX", "PH", "PSX", "CL", "ITW", "FCX", "PYPL", "CSX", "BDX", "MCK", "ABNB", "APH", "TT", "TDG", "USB", "GD", "ORLY", "EMR", "HCA", "NOC", "PNC", "PCAR", "AON", "FDX", "PXD", "NXPI", "MAR", "MCO", "VLO", "CEG", "CTAS", "MSI", "ROP", "ECL", "NSC", "EW", "COF", "AIG", "DXCM", "HLT", "AZO", "APD", "F", "TRV", "AJG", "ADSK", "TFC", "GM", "WELL", "MMM", "NUE", "SPG", "CPRT", "CARR", "MCHP", "URI", "ROST", "WMB", "DHI", "SMCI", "OKE", "PSA", "NEM", "OXY", "MET", "AFL", "ALL", "TEL", "GWW", "SRE", "O", "AEP", "IQV", "JCI", "AMP", "FTNT", "CCI", "MSCI", "DLR", "FAST", "FIS", "BK", "HES", "STZ", "IDXX", "KMB", "A", "DOW", "AME", "PRU", "LULU", "LEN", "MNST", "CMI", "D", "CTVA", "ODFL", "OTIS", "COR", "PAYX", "LHX", "GIS", "HUM", "CNC", "SYY", "RSG", "MLM", "CSGP", "PWR", "IR", "YUM", "EXC", "GEHC", "FANG", "IT", "HAL", "KR", "PCG", "VMC", "CTSH", "KMI", "GEV", "ACGL", "MRNA", "KVUE", "DG", "BKR", "DVN", "CDW", "EL", "ADM", "GPN", "PEG", "PPG", "VRSK", "DD", "RCL", "MPWR", "ROK", "KDP", "EA", "EFX", "EXR", "DFS", "ED", "HIG", "VICI", "FICO", "XYL", "DAL", "ANSS", "XEL", "BIIB", "FTV", "ON", "KHC", "HSY", "WST", "CBRE", "MTD", "KEYS", "WTW", "RMD", "EIX", "CHTR", "TSCO", "CAH", "WAB", "EBAY", "DLTR", "ZBH", "LYB", "TROW", "AVB", "HWM", "TRGP", "WEC", "HPQ", "WY", "NVR", "CHD", "PHM", "BLDR", "FITB", "DOV", "GLW", "RJF", "TTWO", "BR", "NDAQ", "STT", "WDC", "MTB", "HPE", "AWK", "IRM", "SBAC", "GRMN", "ALGN", "DECK", "DTE", "STLD", "ETR", "HUBB", "ULTA", "PTC", "MOH", "CPAY", "NTAP", "AXON", "EQR", "IFF", "APTV", "BAX", "GPC", "CTRA", "STE", "BALL", "ES", "ILMN", "INVH", "BRO", "PPL", "HBAN", "WAT", "FE", "ARE", "COO", "TDY", "LVS", "CBOE", "VLTO", "FSLR", "CINF", "AEE", "TXT", "MKC", "RF", "WBD", "DRI", "PFG", "J", "OMC", "NTRS", "HOLX", "IEX", "CLX", "CNP", "LH", "JBL", "WRB", "LDOS", "AVY", "EXPE", "SYF", "DPZ", "TYL", "VTR", "MAS", "ATO", "CMS", "MRO", "STX", "EXPD", "PKG", "LUV", "TSN", "FDS", "NRG", "SWKS", "VRSN", "TER", "EG", "CE", "CFG", "AKAM", "JBHT", "CCL", "ENPH", "ESS", "BBY", "SNA", "TRMB", "ALB", "BG", "EPAM", "MAA", "POOL", "CF", "ZBRA", "K", "EQT", "CAG", "SWK", "NDSN", "LYV", "DGX", "HST", "KEY", "UAL", "VTRS", "L", "LKQ", "WBA", "PNR", "DOC", "IP", "AMCR", "KMX", "RVTY", "CRL", "MGM", "ROL", "GEN", "JKHY", "WRK", "LNT", "KIM", "TAP", "AES", "EVRG", "IPG", "EMN", "SJM", "PODD", "JNPR", "ALLE", "FFIV", "HII", "UDR", "LW", "QRVO", "NI", "CPT", "TECH", "APA", "AOS", "BBWI", "MOS", "UHS", "CTLT", "INCY", "TFX", "WYNN", "HRL", "TPR", "PAYC", "NWSA", "REG", "DAY", "AIZ", "HSIC", "SOLV", "MTCH", "GL", "BF.B", "CZR", "AAL", "BXP", "CPB", "MKTX", "CHRW", "PNW", "GNRC", "BWA", "NCLH", "RHI", "ETSY", "FOXA", "BEN", "IVZ", "FMC", "FRT", "HAS", "DVA", "CMA", "BIO", "RL", "MHK",]

market_index = "^GSPC"

# Iterar sobre cada año desde 2000 hasta 2023
for year in range(1995, 2000):
    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"

    # Descarga de datos del mercado
    market_data = descargar_datos(market_index, start_date, end_date)

    if market_data is None or market_data.empty:
        print(f"No se obtuvieron datos del mercado para el año {year}")
        continue

    # Crear un DataFrame para almacenar los resultados de cada ticker
    resultados_anuales = pd.DataFrame()

    for ticker in tickers:
        # Descarga de datos del ticker
        ticker_data = descargar_datos(ticker, start_date, end_date)

        if ticker_data is None or ticker_data.empty:
            print(f"No se obtuvieron datos para {ticker} en el año {year}")
            continue

        # Calcular el modelo de regresión
        resultados = calcular_modelo(ticker_data, market_data)

        if resultados:
            resultados_anuales.loc[ticker, 'Const'] = resultados['Const']
            resultados_anuales.loc[ticker, 'P-valor Const'] = resultados['P-valor Const']
            resultados_anuales.loc[ticker, 'Beta'] = resultados['Beta']
            resultados_anuales.loc[ticker, 'P-valor Beta'] = resultados['P-valor Beta']
            resultados_anuales.loc[ticker, 'R^2'] = resultados['R^2']
        else:
            print(f"No se pudo calcular el modelo para {ticker} en el año {year}")

    if not resultados_anuales.empty:
        # Guardar los resultados en un archivo CSV
        resultados_anuales.to_csv(f"resultados_{year}.csv")
        print(f"Resultados guardados para el año {year}")
    else:
        print(f"No se obtuvieron resultados válidos para el año {year}")

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['NVDA']: Exception("%ticker%: Data doesn't exist for startDate = 788936400, endDate = 820386000")


No se obtuvieron datos para NVDA en el año 1995


[*********************100%%**********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AMZN']: Exception("%ticker%: Data doesn't exist for startDate = 788936400, endDate = 820386000")


No se obtuvieron datos para AMZN en el año 1995


[*********************100%%**********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['META']: Exception("%ticker%: Data doesn't exist for startDate = 788936400, endDate = 820386000")


...

No se obtuvieron datos para ETSY en el año 1999


[*********************100%%**********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FOXA']: Exception("%ticker%: Data doesn't exist for startDate = 915166800, endDate = 946616400")
[*********************100%%**********************]  1 of 1 completed


No se obtuvieron datos para FOXA en el año 1999


[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed

Resultados guardados para el año 1999


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os

# Datos de ownership desde 1998 hasta 2023 (con los primeros dos años agregados)
ownership_percentages = [5.2, 6.7, 7.1, 7.7, 8.1, 9.1, 9.6, 9.4, 9.8, 10.8, 12.3, 12.9, 13.2,
    13.6, 14.5, 15.6, 16.5, 16.8, 18.3, 19.5, 20.5, 21.5, 21.1, 21.9, 22.4, 23.1]

# Convertir los porcentajes de ownership a formato decimal
ownership = [x / 100 for x in ownership_percentages]

# Datos de synchronicity
synchronicity = [0.0841186, 0.093674752, 0.10368316, 0.173120673, 0.302341488,
    0.286091414, 0.227016301, 0.231019466, 0.219230048, 0.344672753,
    0.508413239, 0.427980246, 0.457417041, 0.566731814, 0.320952029,
    0.313692786, 0.325876828, 0.399473941, 0.320766834, 0.148923032,
    0.364146572, 0.276907592, 0.538734399, 0.236787372, 0.427625857,
    0.236661531]

# Alinear años para ambos conjuntos de datos (1998-2023)
years = range(1998, 2024)

# Crear el DataFrame para el análisis
data = pd.DataFrame({
    'Year': years,
    '% De propiedad sobre el S&p 500': ownership,
    'Sincronicidad': synchronicity
})

# Ajustar un modelo de regresión lineal usando ownership como variable independiente
import statsmodels.api as sm

X = sm.add_constant(data['% De propiedad sobre el S&p 500'])  # añade constante
y = data['Sincronicidad']
model = sm.OLS(y, X).fit()

result = model.summary()

print(result)

                            OLS Regression Results                            
Dep. Variable:          Sincronicidad   R-squared:                       0.155
Model:                            OLS   Adj. R-squared:                  0.120
Method:                 Least Squares   F-statistic:                     4.396
Date:                Thu, 16 May 2024   Prob (F-statistic):             0.0467
Time:                        19:01:20   Log-Likelihood:                 18.381
No. Observations:                  26   AIC:                            -32.76
Df Residuals:                      24   BIC:                            -30.25
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const     

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm

def descargar_datos(tickers, start_date, end_date):
    """Descarga los datos ajustados al cierre y maneja errores individuales de ticker."""
    datos_list = []  # Lista para almacenar los datos de cada ticker
    for ticker in tickers:
        try:
            ticker_data = yf.download(ticker, start=start_date, end=end_date)['Adj Close']
            ticker_data.name = ticker  # Asegúrate de que la serie tiene el nombre correcto
            datos_list.append(ticker_data)
        except Exception as e:
            print(f"Error descargando datos para {ticker}: {e}")
    # Concatena todas las series en un DataFrame si la lista no está vacía
    if datos_list:
        datos = pd.concat(datos_list, axis=1)
    else:
        datos = pd.DataFrame()
    return datos

def calcular_modelo(datos, market_index):
    """Calcula el modelo de regresión para cada ticker y devuelve un DataFrame con los resultados."""
    resultados = pd.DataFrame()
    log_returns = np.log(datos / datos.shift(1))
    log_returns.dropna(inplace=True)

    for ticker in datos.columns:
        if ticker != market_index:
            try:
                X = sm.add_constant(log_returns[market_index])  # Mercado como variable independiente
                y = log_returns[ticker]  # Retornos del ticker como variable dependiente
                modelo = sm.OLS(y, X).fit()
                resultados.loc[ticker, 'Const'] = modelo.params['const']
                resultados.loc[ticker, 'P-valor Const'] = modelo.pvalues['const']
                resultados.loc[ticker, 'Beta'] = modelo.params[market_index]
                resultados.loc[ticker, 'P-valor Beta'] = modelo.pvalues[market_index]
                resultados.loc[ticker, 'R^2'] = modelo.rsquared
            except Exception as e:
                print(f"Error al ajustar el modelo para {ticker}: {e}")
    return resultados

# Define la lista de tickers y el índice del mercado
tickers = ["MSFT", "AAPL", "NVDA", "AMZN", "META", "GOOGL", "GOOG", "BRK.B", "LLY", "AVGO", "JPM", "TSLA", "XOM", "V", "UNH", "MA", "PG", "JNJ", "HD", "MRK", "COST", "ABBV", "CRM", "CVX", "AMD", "NFLX", "BAC", "WMT", "PEP", "KO", "LIN", "TMO", "ADBE", "DIS", "ACN", "WFC", "ORCL", "CSCO", "MCD", "QCOM", "ABT", "CAT", "INTU", "AMAT", "IBM", "VZ", "GE", "CMCSA", "NOW", "INTC", "DHR", "COP", "UBER", "TXN", "PFE", "UNP", "AMGN", "PM", "LOW", "SPGI", "ISRG", "MU", "RTX", "GS", "NEE", "HON", "ETN", "AXP", "LRCX", "BKNG", "PGR", "T", "ELV", "SYK", "C", "MS", "PLD", "BLK", "MDT", "TJX", "NKE", "UPS", "SCHW", "DE", "CI", "BA", "VRTX", "BMY", "CB", "ADP", "MMC", "BSX", "REGN", "SBUX", "ADI", "LMT", "FI", "KLAC", "CVS", "BX", "MDLZ", "AMT", "SNPS", "GILD", "PANW", "CDNS", "TMUS", "CMG", "MPC", "EOG", "ICE", "TGT", "SHW", "SLB", "CME", "SO", "ZTS", "WM", "ANET", "DUK", "MO", "EQIX", "PH", "PSX", "CL", "ITW", "FCX", "PYPL", "CSX", "BDX", "MCK", "ABNB", "APH", "TT", "TDG", "USB", "GD", "ORLY", "EMR", "HCA", "NOC", "PNC", "PCAR", "AON", "FDX", "PXD", "NXPI", "MAR", "MCO", "VLO", "CEG", "CTAS", "MSI", "ROP", "ECL", "NSC", "EW", "COF", "AIG", "DXCM", "HLT", "AZO", "APD", "F", "TRV", "AJG", "ADSK", "TFC", "GM", "WELL", "MMM", "NUE", "SPG", "CPRT", "CARR", "MCHP", "URI", "ROST", "WMB", "DHI", "SMCI", "OKE", "PSA", "NEM", "OXY", "MET", "AFL", "ALL", "TEL", "GWW", "SRE", "O", "AEP", "IQV", "JCI", "AMP", "FTNT", "CCI", "MSCI", "DLR", "FAST", "FIS", "BK", "HES", "STZ", "IDXX", "KMB", "A", "DOW", "AME", "PRU", "LULU", "LEN", "MNST", "CMI", "D", "CTVA", "ODFL", "OTIS", "COR", "PAYX", "LHX", "GIS", "HUM", "CNC", "SYY", "RSG", "MLM", "CSGP", "PWR", "IR", "YUM", "EXC", "GEHC", "FANG", "IT", "HAL", "KR", "PCG", "VMC", "CTSH", "KMI", "GEV", "ACGL", "MRNA", "KVUE", "DG", "BKR", "DVN", "CDW", "EL", "ADM", "GPN", "PEG", "PPG", "VRSK", "DD", "RCL", "MPWR", "ROK", "KDP", "EA", "EFX", "EXR", "DFS", "ED", "HIG", "VICI", "FICO", "XYL", "DAL", "ANSS", "XEL", "BIIB", "FTV", "ON", "KHC", "HSY", "WST", "CBRE", "MTD", "KEYS", "WTW", "RMD", "EIX", "CHTR", "TSCO", "CAH", "WAB", "EBAY", "DLTR", "ZBH", "LYB", "TROW", "AVB", "HWM", "TRGP", "WEC", "HPQ", "WY", "NVR", "CHD", "PHM", "BLDR", "FITB", "DOV", "GLW", "RJF", "TTWO", "BR", "NDAQ", "STT", "WDC", "MTB", "HPE", "AWK", "IRM", "SBAC", "GRMN", "ALGN", "DECK", "DTE", "STLD", "ETR", "HUBB", "ULTA", "PTC", "MOH", "CPAY", "NTAP", "AXON", "EQR", "IFF", "APTV", "BAX", "GPC", "CTRA", "STE", "BALL", "ES", "ILMN", "INVH", "BRO", "PPL", "HBAN", "WAT", "FE", "ARE", "COO", "TDY", "LVS", "CBOE", "VLTO", "FSLR", "CINF", "AEE", "TXT", "MKC", "RF", "WBD", "DRI", "PFG", "J", "OMC", "NTRS", "HOLX", "IEX", "CLX", "CNP", "LH", "JBL", "WRB", "LDOS", "AVY", "EXPE", "SYF", "DPZ", "TYL", "VTR", "MAS", "ATO", "CMS", "MRO", "STX", "EXPD", "PKG", "LUV", "TSN", "FDS", "NRG", "SWKS", "VRSN", "TER", "EG", "CE", "CFG", "AKAM", "JBHT", "CCL", "ENPH", "ESS", "BBY", "SNA", "TRMB", "ALB", "BG", "EPAM", "MAA", "POOL", "CF", "ZBRA", "K", "EQT", "CAG", "SWK", "NDSN", "LYV", "DGX", "HST", "KEY", "UAL", "VTRS", "L", "LKQ", "WBA", "PNR", "DOC", "IP", "AMCR", "KMX", "RVTY", "CRL", "MGM", "ROL", "GEN", "JKHY", "WRK", "LNT", "KIM", "TAP", "AES", "EVRG", "IPG", "EMN", "SJM", "PODD", "JNPR", "ALLE", "FFIV", "HII", "UDR", "LW", "QRVO", "NI", "CPT", "TECH", "APA", "AOS", "BBWI", "MOS", "UHS", "CTLT", "INCY", "TFX", "WYNN", "HRL", "TPR", "PAYC", "NWSA", "REG", "DAY", "AIZ", "HSIC", "SOLV", "MTCH", "GL", "BF.B", "CZR", "AAL", "BXP", "CPB", "MKTX", "CHRW", "PNW", "GNRC", "BWA", "NCLH", "RHI", "ETSY", "FOXA", "BEN", "IVZ", "FMC", "FRT", "HAS", "DVA", "CMA", "BIO", "RL", "MHK",]

market_index = "^GSPC"

# Descarga de datos con manejo de errores
data = descargar_datos(tickers + [market_index], "2020-01-01", "2023-01-01")

# Asegurarse de que los datos se hayan descargado correctamente antes de continuar
if not data.empty:
    resultados = calcular_modelo(data, market_index)
    print(resultados)
else:
    print("No se obtuvieron datos válidos, revisa la lista de tickers y la conexión a internet.")

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['BRK.B']: Exception('%ticker%: No timezone found, symbol may be delisted')
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%*******************

Error al ajustar el modelo para MSFT: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para AAPL: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para NVDA: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para AMZN: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para META: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para GOOGL: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para GOOG: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para BRK.B: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para LLY: zero-size array to reduction operation maximum which has no identity
Error al ajustar el modelo para AVGO

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm

def descargar_datos(ticker, start_date, end_date):
    try:
        datos = yf.download(ticker, start=start_date, end=end_date)['Adj Close']
        datos.name = ticker
        return datos
    except Exception as e:
        print(f"Error descargando datos para {ticker}: {e}")
        return pd.Series()

def calcular_modelo(y, X):
    try:
        X = sm.add_constant(X)  # Añadir una constante al modelo
        modelo = sm.OLS(y, X).fit()
        return {
            'Const': modelo.params['const'],
            'P-valor Const': modelo.pvalues['const'],
            'Beta': modelo.params[X.columns[1]],
            'P-valor Beta': modelo.pvalues[X.columns[1]],
            'R^2': modelo.rsquared
        }
    except Exception as e:
        print(f"Error al ajustar el modelo: {e}")
        return {}

# Define la lista de tickers y el índice del mercado
tickers = ["MSFT", "AAPL", "NVDA", "AMZN", "META", "GOOGL", "GOOG", "BRK.B", "LLY", "AVGO", "JPM", "TSLA", "XOM", "V", "UNH", "MA", "PG", "JNJ", "HD", "MRK", "COST", "ABBV", "CRM", "CVX", "AMD", "NFLX", "BAC", "WMT", "PEP", "KO", "LIN", "TMO", "ADBE", "DIS", "ACN", "WFC", "ORCL", "CSCO", "MCD", "QCOM", "ABT", "CAT", "INTU", "AMAT", "IBM", "VZ", "GE", "CMCSA", "NOW", "INTC", "DHR", "COP", "UBER", "TXN", "PFE", "UNP", "AMGN", "PM", "LOW", "SPGI", "ISRG", "MU", "RTX", "GS", "NEE", "HON", "ETN", "AXP", "LRCX", "BKNG", "PGR", "T", "ELV", "SYK", "C", "MS", "PLD", "BLK", "MDT", "TJX", "NKE", "UPS", "SCHW", "DE", "CI", "BA", "VRTX", "BMY", "CB", "ADP", "MMC", "BSX", "REGN", "SBUX", "ADI", "LMT", "FI", "KLAC", "CVS", "BX", "MDLZ", "AMT", "SNPS", "GILD", "PANW", "CDNS", "TMUS", "CMG", "MPC", "EOG", "ICE", "TGT", "SHW", "SLB", "CME", "SO", "ZTS", "WM", "ANET", "DUK", "MO", "EQIX", "PH", "PSX", "CL", "ITW", "FCX", "PYPL", "CSX", "BDX", "MCK", "ABNB", "APH", "TT", "TDG", "USB", "GD", "ORLY", "EMR", "HCA", "NOC", "PNC", "PCAR", "AON", "FDX", "PXD", "NXPI", "MAR", "MCO", "VLO", "CEG", "CTAS", "MSI", "ROP", "ECL", "NSC", "EW", "COF", "AIG", "DXCM", "HLT", "AZO", "APD", "F", "TRV", "AJG", "ADSK", "TFC", "GM", "WELL", "MMM", "NUE", "SPG", "CPRT", "CARR", "MCHP", "URI", "ROST", "WMB", "DHI", "SMCI", "OKE", "PSA", "NEM", "OXY", "MET", "AFL", "ALL", "TEL", "GWW", "SRE", "O", "AEP", "IQV", "JCI", "AMP", "FTNT", "CCI", "MSCI", "DLR", "FAST", "FIS", "BK", "HES", "STZ", "IDXX", "KMB", "A", "DOW", "AME", "PRU", "LULU", "LEN", "MNST", "CMI", "D", "CTVA", "ODFL", "OTIS", "COR", "PAYX", "LHX", "GIS", "HUM", "CNC", "SYY", "RSG", "MLM", "CSGP", "PWR", "IR", "YUM", "EXC", "GEHC", "FANG", "IT", "HAL", "KR", "PCG", "VMC", "CTSH", "KMI", "GEV", "ACGL", "MRNA", "KVUE", "DG", "BKR", "DVN", "CDW", "EL", "ADM", "GPN", "PEG", "PPG", "VRSK", "DD", "RCL", "MPWR", "ROK", "KDP", "EA", "EFX", "EXR", "DFS", "ED", "HIG", "VICI", "FICO", "XYL", "DAL", "ANSS", "XEL", "BIIB", "FTV", "ON", "KHC", "HSY", "WST", "CBRE", "MTD", "KEYS", "WTW", "RMD", "EIX", "CHTR", "TSCO", "CAH", "WAB", "EBAY", "DLTR", "ZBH", "LYB", "TROW", "AVB", "HWM", "TRGP", "WEC", "HPQ", "WY", "NVR", "CHD", "PHM", "BLDR", "FITB", "DOV", "GLW", "RJF", "TTWO", "BR", "NDAQ", "STT", "WDC", "MTB", "HPE", "AWK", "IRM", "SBAC", "GRMN", "ALGN", "DECK", "DTE", "STLD", "ETR", "HUBB", "ULTA", "PTC", "MOH", "CPAY", "NTAP", "AXON", "EQR", "IFF", "APTV", "BAX", "GPC", "CTRA", "STE", "BALL", "ES", "ILMN", "INVH", "BRO", "PPL", "HBAN", "WAT", "FE", "ARE", "COO", "TDY", "LVS", "CBOE", "VLTO", "FSLR", "CINF", "AEE", "TXT", "MKC", "RF", "WBD", "DRI", "PFG", "J", "OMC", "NTRS", "HOLX", "IEX", "CLX", "CNP", "LH", "JBL", "WRB", "LDOS", "AVY", "EXPE", "SYF", "DPZ", "TYL", "VTR", "MAS", "ATO", "CMS", "MRO", "STX", "EXPD", "PKG", "LUV", "TSN", "FDS", "NRG", "SWKS", "VRSN", "TER", "EG", "CE", "CFG", "AKAM", "JBHT", "CCL", "ENPH", "ESS", "BBY", "SNA", "TRMB", "ALB", "BG", "EPAM", "MAA", "POOL", "CF", "ZBRA", "K", "EQT", "CAG", "SWK", "NDSN", "LYV", "DGX", "HST", "KEY", "UAL", "VTRS", "L", "LKQ", "WBA", "PNR", "DOC", "IP", "AMCR", "KMX", "RVTY", "CRL", "MGM", "ROL", "GEN", "JKHY", "WRK", "LNT", "KIM", "TAP", "AES", "EVRG", "IPG", "EMN", "SJM", "PODD", "JNPR", "ALLE", "FFIV", "HII", "UDR", "LW", "QRVO", "NI", "CPT", "TECH", "APA", "AOS", "BBWI", "MOS", "UHS", "CTLT", "INCY", "TFX", "WYNN", "HRL", "TPR", "PAYC", "NWSA", "REG", "DAY", "AIZ", "HSIC", "SOLV", "MTCH", "GL", "BF.B", "CZR", "AAL", "BXP", "CPB", "MKTX", "CHRW", "PNW", "GNRC", "BWA", "NCLH", "RHI", "ETSY", "FOXA", "BEN", "IVZ", "FMC", "FRT", "HAS", "DVA", "CMA", "BIO", "RL", "MHK",]

market_index = "^GSPC"
start_date = "1996-01-01"
end_date = "1992-01-01"

market_data = descargar_datos(market_index, start_date, end_date)
resultados = pd.DataFrame()

for ticker in tickers:
    ticker_data = descargar_datos(ticker, start_date, end_date)
    if not ticker_data.empty and not market_data.empty:
        log_returns_ticker = np.log(ticker_data / ticker_data.shift(1))
        log_returns_market = np.log(market_data / market_data.shift(1))
        common_index = log_returns_ticker.dropna().index.intersection(log_returns_market.dropna().index)
        y = log_returns_ticker.loc[common_index]
        X = log_returns_market.loc[common_index]
        if not y.empty and not X.empty:
            resultado = calcular_modelo(y, X)
            df_resultado = pd.DataFrame([resultado], index=[ticker])
            resultados = pd.concat([resultados, df_resultado])
        else:
            print(f"No hay suficientes datos para calcular el modelo para {ticker}.")
    else:
        print(f"No se pudieron obtener datos para {ticker}.")

resultados.to_csv('/content/resultados.csv', index=True)

print(resultados)

[*********************100%%**********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['^GSPC']: Exception('%ticker%: Invalid input - start date cannot be after end date. startDate = 820472400, endDate = 694242000')
[*********************100%%**********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['MSFT']: Exception('%ticker%: Invalid input - start date cannot be after end date. startDate = 820472400, endDate = 694242000')


No se pudieron obtener datos para MSFT.


[*********************100%%**********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AAPL']: Exception('%ticker%: Invalid input - start date cannot be after end date. startDate = 820472400, endDate = 694242000')


No se pudieron obtener datos para AAPL.


[*********************100%%**********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['NVDA']: Exception('%ticker%: Invalid input - start date cannot be after end date. startDate = 820472400, endDate = 694242000')


...

No se pudieron obtener datos para BIO.


[*********************100%%**********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['RL']: Exception('%ticker%: Invalid input - start date cannot be after end date. startDate = 820472400, endDate = 694242000')


No se pudieron obtener datos para RL.


[*********************100%%**********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['MHK']: Exception('%ticker%: Invalid input - start date cannot be after end date. startDate = 820472400, endDate = 694242000')


No se pudieron obtener datos para MHK.
Empty DataFrame
Columns: []
Index: []


In [ ]:
import yfinance as yf

msft = yf.Ticker("AAPL")


hist = msft.history(period="1mo")

mholders = msft.major_holders

instholders = msft.institutional_holders

print(mholders)
print(instholders)



# show meta information about the history (requires history() to be called first)
msft.history_metadata

Breakdown                          Value
insidersPercentHeld              0.05978
institutionsPercentHeld          0.57335
institutionsFloatPercentHeld     0.60980
institutionsCount             6242.00000
  Date Reported                             Holder  pctHeld      Shares  \
0    2023-06-30                 Vanguard Group Inc   0.0834  1303688506   
1    2023-06-30                     Blackrock Inc.   0.0665  1039640859   
2    2023-06-30            Berkshire Hathaway, Inc   0.0586   915560382   
3    2023-06-30           State Street Corporation   0.0370   578897858   
4    2023-06-30                           FMR, LLC   0.0196   307066638   
5    2023-06-30      Geode Capital Management, LLC   0.0186   291538165   
6    2023-06-30      Price (T.Rowe) Associates Inc   0.0145   226650943   
7    2023-06-30                     Morgan Stanley   0.0131   204714950   
8    2022-12-31  Norges Bank Investment Management   0.0107   167374278   
9    2023-06-30         Northern Trust Corpor

{'currency': 'USD',
 'symbol': 'AAPL',
 'exchangeName': 'NMS',
 'fullExchangeName': 'NasdaqGS',
 'instrumentType': 'EQUITY',
 'firstTradeDate': 345479400,
 'regularMarketTime': 1715371201,
 'hasPrePostMarketData': True,
 'gmtoffset': -14400,
 'timezone': 'EDT',
 'exchangeTimezoneName': 'America/New_York',
 'regularMarketPrice': 183.05,
 'fiftyTwoWeekHigh': 185.09,
 'fiftyTwoWeekLow': 182.13,
 'regularMarketDayHigh': 185.09,
 'regularMarketDayLow': 182.13,
 'regularMarketVolume': 49014221,
 'chartPreviousClose': 167.78,
 'priceHint': 2,
 'currentTradingPeriod': {'pre': {'timezone': 'EDT',
   'end': 1715347800,
   'start': 1715328000,
   'gmtoffset': -14400},
  'regular': {'timezone': 'EDT',
   'end': 1715371200,
   'start': 1715347800,
   'gmtoffset': -14400},
  'post': {'timezone': 'EDT',
   'end': 1715385600,
   'start': 1715371200,
   'gmtoffset': -14400}},
 'dataGranularity': '1d',
 'range': '1mo',
 'validRanges': ['1d',
  '5d',
  '1mo',
  '3mo',
  '6mo',
  '1y',
  '2y',
  '5y',
  '